In [ ]:
%pip install -q "transformers>=4.46,<6" "accelerate>=1.0,<2"


In [ ]:
import csv
import os
import random

import numpy as np
import pandas as pd
import torch
from bs4 import BeautifulSoup
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Kaggle 上写 /kaggle/working，本地写 ./outputs
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('PyTorch 版本：', torch.__version__)
print('训练设备：', device)
print('输出目录：', OUTPUT_DIR)


In [ ]:
# 数据文件可能在这几个位置：环境变量、Kaggle 输入目录、当前目录
DATA_DIR_CANDIDATES = [
    os.environ.get('WORD2VEC_DATA_DIR', ''),
    '/kaggle/input/competitions/word2vec-nlp-tutorial',
    '/kaggle/input/word2vec-nlp-tutorial',
    os.path.abspath('.'),
    os.path.abspath('./data'),
]

TRAIN_FILES = ('labeledTrainData.tsv', 'labeledTrainData.tsv.zip')
TEST_FILES = ('testData.tsv', 'testData.tsv.zip')


def find_data_file(filenames):
    """按候选目录顺序返回第一个存在的数据文件。"""
    for directory in DATA_DIR_CANDIDATES:
        if not directory:
            continue
        for filename in filenames:
            path = os.path.join(directory, filename)
            if os.path.exists(path):
                return path
    return None


def read_review_tsv(path, num_columns):
    """读带 review 列的 TSV；引号不规范时退回 QUOTE_NONE（等价教程里的 quoting=3）。"""
    data = pd.read_csv(path, sep='\t')
    if data.shape[1] != num_columns:
        data = pd.read_csv(path, sep='\t', quoting=csv.QUOTE_NONE)
    return data


train_path = find_data_file(TRAIN_FILES)
test_path = find_data_file(TEST_FILES)

if train_path is None or test_path is None:
    # 没挂载数据集时用 kagglehub 现场下载
    try:
        import kagglehub

        dataset_dir = kagglehub.competition_download('word2vec-nlp-tutorial')
        DATA_DIR_CANDIDATES.insert(0, dataset_dir)
        train_path = find_data_file(TRAIN_FILES)
        test_path = find_data_file(TEST_FILES)
    except Exception as error:
        print('kagglehub 下载失败：', error)

assert train_path is not None, '没找到 labeledTrainData.tsv(.zip)，请放到当前目录或设置 WORD2VEC_DATA_DIR'
assert test_path is not None, '没找到 testData.tsv(.zip)，请放到当前目录或设置 WORD2VEC_DATA_DIR'

print('训练集文件：', train_path)
print('测试集文件：', test_path)

train = read_review_tsv(train_path, 3)
test = read_review_tsv(test_path, 2)

assert list(train.columns) == ['id', 'sentiment', 'review'], train.columns
assert list(test.columns) == ['id', 'review'], test.columns
assert len(test) == test['id'].nunique(), '测试集 id 有重复，读入可能出错'
assert train['sentiment'].isin([0, 1]).all(), '标签不是 0/1'

print('训练集：', train.shape)
print('测试集：', test.shape)
print(train.head())


In [ ]:
def save_submission(predictions, filename):

    submission = pd.DataFrame({
        'id': test['id'].values,
        'sentiment': np.asarray(predictions, dtype=np.float32)
    })

    # 检查提交文件
    assert len(submission) == len(test)
    assert submission['id'].is_unique
    assert submission['id'].tolist() == test['id'].tolist()
    assert np.isfinite(submission['sentiment']).all()
    assert submission['sentiment'].between(0, 1).all()

    path = os.path.join(OUTPUT_DIR, filename)

    submission.to_csv(path, index=False)

    print('文件已生成：', path)
    print('文件形状：', submission.shape)
    print(submission.head())

    return submission


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
    EarlyStoppingCallback, set_seed
)

ROBERTA_TRAINER_NAME = 'FacebookAI/roberta-base'
ROBERTA_TRAINER_MAX_LEN = 256
ROBERTA_TRAINER_BATCH_SIZE = 8

roberta_trainer_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_TRAINER_NAME)


class RobertaTrainerDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = int(self.labels[index])
        return item


def encode_roberta_trainer_reviews(reviews):
    texts = [
        BeautifulSoup(str(review), 'html.parser').get_text(' ')
        for review in reviews
    ]
    return roberta_trainer_tokenizer(
        texts,
        truncation=True,
        max_length=ROBERTA_TRAINER_MAX_LEN,
        return_token_type_ids=False
    )


roberta_trainer_train_idx, roberta_trainer_val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.2,
    random_state=SEED,
    stratify=train['sentiment'].values
)
roberta_trainer_labels = train['sentiment'].values

roberta_trainer_train_dataset = RobertaTrainerDataset(
    encode_roberta_trainer_reviews(train['review'].iloc[roberta_trainer_train_idx]),
    roberta_trainer_labels[roberta_trainer_train_idx]
)
roberta_trainer_val_dataset = RobertaTrainerDataset(
    encode_roberta_trainer_reviews(train['review'].iloc[roberta_trainer_val_idx]),
    roberta_trainer_labels[roberta_trainer_val_idx]
)
roberta_trainer_test_dataset = RobertaTrainerDataset(
    encode_roberta_trainer_reviews(test['review'])
)

print('训练集：', len(roberta_trainer_train_dataset))
print('验证集：', len(roberta_trainer_val_dataset))
print('测试集：', len(roberta_trainer_test_dataset))
print('样本编码长度示例：', len(roberta_trainer_train_dataset[0]['input_ids']))


In [ ]:
def compute_roberta_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    return {'auc': roc_auc_score(labels, probabilities)}


set_seed(SEED)
roberta_trainer_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_TRAINER_NAME,
    num_labels=2
)

roberta_trainer_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, 'roberta_trainer_checkpoints'),
    learning_rate=2e-5,
    per_device_train_batch_size=ROBERTA_TRAINER_BATCH_SIZE,
    per_device_eval_batch_size=ROBERTA_TRAINER_BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,
    optim='adamw_torch',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    disable_tqdm=True,
    seed=SEED
)

roberta_trainer = Trainer(
    model=roberta_trainer_model,
    args=roberta_trainer_args,
    train_dataset=roberta_trainer_train_dataset,
    eval_dataset=roberta_trainer_val_dataset,
    processing_class=roberta_trainer_tokenizer,
    data_collator=DataCollatorWithPadding(
        tokenizer=roberta_trainer_tokenizer,
        pad_to_multiple_of=8
    ),
    compute_metrics=compute_roberta_trainer_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
roberta_trainer.train()
print('RoBERTa Trainer 最佳验证集 AUC：', roberta_trainer.state.best_metric)

roberta_trainer.save_model(os.path.join(OUTPUT_DIR, 'roberta_trainer_best'))
roberta_trainer_tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'roberta_trainer_best'))


In [ ]:
roberta_trainer_test_output = roberta_trainer.predict(roberta_trainer_test_dataset)
roberta_trainer_predictions = torch.softmax(
    torch.from_numpy(roberta_trainer_test_output.predictions), dim=-1
).numpy()[:, 1]

roberta_trainer_submission = save_submission(
    roberta_trainer_predictions,
    'submission_roberta_trainer.csv'
)


In [ ]:
submission_path = os.path.join(OUTPUT_DIR, 'submission_roberta_trainer.csv')

assert os.path.exists(submission_path), f'提交文件不存在：{submission_path}'
assert len(roberta_trainer_submission) == len(test)

print('最佳验证集 AUC：', roberta_trainer.state.best_metric)
print('提交文件：', submission_path)
print('提交行数：', len(roberta_trainer_submission))
print('预测概率统计：')
print(roberta_trainer_submission['sentiment'].describe())
print('预测为正面（>0.5）的比例：', float((roberta_trainer_submission['sentiment'] > 0.5).mean()))
